# Frequency Law v9.0 — GitHub Render-Safe Edition

**Author:** Christian Berrang  
**DOI:** `10.5281/zenodo.17874830`

> "The equations stay the same. The direction of reading changes."

This notebook is cleaned for GitHub rendering:

- no `%matplotlib inline`
- no stored outputs
- no Colab metadata
- no widget / custom MIME outputs
- no `display()` dependency


In [ ]:
import json
import sys
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

# Physical constants
h = 6.62607015e-34      # Planck constant [J*s]
c = 299_792_458         # Speed of light [m/s]
c2 = c**2
eV = 1.602176634e-19    # Electron volt [J]
G = 6.6743e-11          # Gravitational constant [m^3 kg^-1 s^-2]
k_B = 1.380649e-23      # Boltzmann constant [J/K]
hbar = h / (2 * np.pi)

print(f"Python: {sys.version.split()[0]}")
print(f"h = {h:.10e} J*s")
print(f"c = {c} m/s")
print(f"hbar = {hbar:.10e} J*s")
print("Constants loaded OK")


## Axioms A0–A6

| ID | Name | Formal | Status |
|---|---|---|---|
| A0 | Null Field | `N := {dPhi=0}` | definition |
| A1 | Frequency is primary | `f [Hz]` | primary |
| A2 | Information | `I ~ dPhi` | primary |
| A3 | Time is emergent | `T = dPhi / f` | derived |
| A4 | Energy is derived | `E = h*f` | derived |
| A5 | Mass = bound frequency | `m = h*f / c^2` | derived |
| A6 | Frequency conservation | `sum h*fi = const` | hypothesis |


In [ ]:
axioms = [
    {"id": "A0", "name": "Null Field", "formal": "N := {dPhi=0}", "status": "definition"},
    {"id": "A1", "name": "Frequency is primary", "formal": "f [Hz]", "status": "primary"},
    {"id": "A2", "name": "Information", "formal": "I ~ dPhi", "status": "primary"},
    {"id": "A3", "name": "Time is emergent", "formal": "T = dPhi / f", "status": "derived"},
    {"id": "A4", "name": "Energy is derived", "formal": "E = h*f", "status": "derived"},
    {"id": "A5", "name": "Mass = bound frequency", "formal": "m = h*f / c^2", "status": "derived"},
    {"id": "A6", "name": "Freq. conservation", "formal": "sum h*fi = const", "status": "hypothesis"},
]

print(pd.DataFrame(axioms).to_string(index=False))


## Core formulas

The numerical identity used for validation is:

`m = h*f / c^2`

In this framework the proposed causal reading is `f -> m`, not `m -> f`.


In [ ]:
def mass_from_frequency(f_hz: float) -> float:
    """Return mass [kg] from frequency [Hz]."""
    return (h * f_hz) / c2


def frequency_from_mass(mass_kg: float) -> float:
    """Return frequency [Hz] from mass [kg]."""
    return (mass_kg * c2) / h


def mev_to_kg(mass_mev: float) -> float:
    """Convert MeV/c^2 to kg."""
    return (mass_mev * 1e6 * eV) / c2


def zitterbewegung_frequency(f_compton: float, topology: str = "Mobius") -> float:
    """Return proposed zitterbewegung frequency."""
    return 2 * f_compton if topology == "Mobius" else f_compton


print("Core formulas loaded OK")


## Particle class and database


In [ ]:
@dataclass
class Particle:
    name: str
    mass_MeV: float
    topology: str
    status: str
    generation: int | None = None

    @property
    def mass_kg(self) -> float:
        return mev_to_kg(self.mass_MeV)

    @property
    def compton_freq(self) -> float:
        return frequency_from_mass(self.mass_kg)

    @property
    def zitter_freq(self) -> float:
        top = "Mobius" if "Mobius" in self.topology else "circle"
        return zitterbewegung_frequency(self.compton_freq, top)


FERMIONS = [
    Particle("Neutrino v1",        0.000002, "Mobius 4pi", "known", 1),
    Particle("Electron e-",        0.511,    "Mobius 4pi", "known", 1),
    Particle("Berrangium Omega",   16.2,     "Mobius 4pi", "PREDICTION"),
    Particle("Muon mu-",           105.7,    "Mobius 4pi", "known", 2),
    Particle("Stoecker Particle",  530.0,    "Mobius 4pi", "PREDICTION"),
    Particle("Proton p",           938.3,    "Mobius 4pi", "known", 1),
    Particle("Tau tau-",           1777.0,   "Mobius 4pi", "known", 3),
    Particle("Bottom quark b",     4180.0,   "Mobius 4pi", "known", 3),
    Particle("Top quark t",        172700.0, "Mobius 4pi", "known", 3),
]

BOSONS = [
    Particle("W Boson",   80400.0,  "circle 2pi", "known"),
    Particle("Higgs H",   125100.0, "circle 2pi", "known"),
]

ALL_PARTICLES = FERMIONS + BOSONS

print(f"Loaded {len(FERMIONS)} fermions, {len(BOSONS)} bosons")
print(f"{sum(1 for p in ALL_PARTICLES if p.status == 'PREDICTION')} predictions included")


## Validation against PDG reference masses

The listed PDG masses are used here as reference values for numerical comparison.


In [ ]:
PDG = {
    "Electron": {"mass_pdg_kg": 9.1093837015e-31, "f_model_hz": 1.2358e20},
    "Proton":   {"mass_pdg_kg": 1.67262192369e-27, "f_model_hz": 2.2687e23},
    "Neutron":  {"mass_pdg_kg": 1.67492749804e-27, "f_model_hz": 2.2718e23},
    "Muon":     {"mass_pdg_kg": 1.88353162e-28, "f_model_hz": 2.5554e22},
    "Higgs":    {"mass_pdg_kg": 2.225e-25, "f_model_hz": 3.018e25},
}

rows = []
for name, vals in PDG.items():
    m_pdg = vals["mass_pdg_kg"]
    f_mod = vals["f_model_hz"]
    m_calc = mass_from_frequency(f_mod)
    dev = abs(m_calc - m_pdg) / m_pdg * 100
    rows.append({
        "Particle": name,
        "f_model (Hz)": f_mod,
        "m_calc (kg)": m_calc,
        "m_PDG (kg)": m_pdg,
        "Deviation %": round(dev, 6),
    })

df_valid = pd.DataFrame(rows)
print(df_valid.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(df_valid["Particle"], df_valid["Deviation %"])
ax.set_ylabel("Deviation [%] log scale")
ax.set_title("Validation: m = h*f / c^2 vs PDG")
ax.set_yscale("log")
ax.grid(axis="y", alpha=0.4)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Pauli Principle as geometry

For identical fermions:

`Psi_total = psi1 - psi2 = 0`

In this interpretation, the exclusion is treated as topologically enforced.


In [ ]:
def pauli_wavefunction(psi1: complex, psi2: complex) -> complex:
    return psi1 + psi2 * np.exp(1j * np.pi)


psi_id = pauli_wavefunction(1.0 + 0j, 1.0 + 0j)
psi_diff = pauli_wavefunction(1.0 + 0j, 0.5 + 0.5j)

print(f"Identical fermions: Psi = {psi_id} |Psi|^2 = {abs(psi_id)**2} -> Forbidden")
print(f"Different fermions: Psi = {psi_diff:.4f} |Psi|^2 = {abs(psi_diff)**2:.4f} -> Allowed")


## Mobius topology and phase cycles


In [ ]:
print("Fermions (Mobius — spin-1/2):")
print(f"  dPhi = 4pi = {4*np.pi:.6f} rad")
print("  After 2pi: psi -> -psi (sign flip)")
print("  After 4pi: psi -> +psi (return)")
print("  -> f_zitter = 2 x f_compton")
print()
print("Bosons (circle — integer spin):")
print(f"  dPhi = 2pi = {2*np.pi:.6f} rad")
print("  After 2pi: psi -> +psi (return)")


## Predictions: Berrangium and Stoecker


In [ ]:
b = next(p for p in FERMIONS if "Berrangium" in p.name)
s = next(p for p in FERMIONS if "Stoecker" in p.name)

print("BERRANGIUM OMEGA")
print(f"  Mass:         {b.mass_MeV} MeV/c^2")
print(f"  Compton freq: {b.compton_freq:.4e} Hz")
print("  Position:     Between electron and muon")
print("  Hint:         X17 anomaly near 17 MeV")
print("  Status:       Open")
print()
print("STOECKER PARTICLE")
print(f"  Mass:         {s.mass_MeV} MeV/c^2")
print(f"  Compton freq: {s.compton_freq:.4e} Hz")
print("  Position:     Between muon and proton")
print("  Hint:         f0(500) resonance range")
print("  Status:       Open")


## Frequency Periodic Table


In [ ]:
pp = sorted([p for p in ALL_PARTICLES if p.mass_MeV > 0], key=lambda p: p.compton_freq)
names = [p.name for p in pp]
freqs = [p.compton_freq for p in pp]

fig, ax = plt.subplots(figsize=(14, 9))
ax.barh(range(len(names)), freqs)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel("Compton Frequency (Hz)")
ax.set_title("Frequency Periodic Table")
ax.set_xscale("log")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()


## Experimental test matrix

| Priority | Test | Method | Status |
|---|---|---|---|
| High | Phase-time `T = dPhi/f` | Mach-Zehnder | Open |
| High | Berrangium at ~16.2 MeV | Accelerator 15–17 MeV | Open |
| High | Stoecker at ~530 MeV | LHCb, BESIII, GlueX | Open |
| Medium | Time dilation `T = dPhi/f` | GPS, atomic clocks | Compatible |
| Exploratory | Zitterbewegung 2x | Electron scattering | Consistent |

All predictions are falsifiable. The gaps are either there or they are not.


## JSON Framework Export


In [ ]:
framework = {
    "framework": "Frequency Law",
    "version": "9.0",
    "doi": "10.5281/zenodo.17874830",
    "causal_direction": "f -> dPhi -> T -> m -> E",
    "axioms": {
        "A0": {"name": "Null Field", "formal": "N := {dPhi=0}", "status": "definition"},
        "A1": {"name": "Frequency primary", "formal": "f [Hz]", "status": "primary"},
        "A2": {"name": "Phase = information", "formal": "I ~ dPhi", "status": "primary"},
        "A3": {"name": "Time emergent", "formal": "T = dPhi / f", "status": "derived"},
        "A4": {"name": "Energy derived", "formal": "E = h*f", "status": "derived"},
        "A5": {"name": "Mass = bound frequency", "formal": "m = h*f / c^2", "status": "derived"},
        "A6": {"name": "Freq. conservation", "formal": "sum h*fi = const", "status": "hypothesis"},
    },
    "falsification_targets": [
        {"name": "Berrangium Omega", "predicted_MeV": 16.2, "search_range_MeV": [15, 17], "status": "open"},
        {"name": "Stoecker Particle", "predicted_MeV": 530.0, "search_range_MeV": [450, 600], "status": "open"},
        {"name": "Phase-time", "formula": "T = dPhi/f", "method": "Mach-Zehnder", "status": "open"},
    ],
}

print(json.dumps(framework, indent=2))


## Summary

| Result | Value |
|---|---|
| Electron Compton frequency | approximately `1.2356 x 10^20 Hz` |
| Zitterbewegung factor | `2x` from Mobius topology |
| Pauli Principle | geometrically interpreted as `Psi_total = 0` |
| Causal direction | `f -> m`, not `m -> f` |
| Berrangium | approximately `16.2 MeV`, search open |
| Stoecker | approximately `530 MeV`, search open |

**Frequency Law v9.0** | Christian Berrang | DOI: `10.5281/zenodo.17874830`
